# Function Call 02 · 三种 Agent 实现对比（原生 SDK / LangChain / DeepAgents）

同一个任务、三条路径：**手工循环 → 框架封装 → 深度智能体**。
这一课的价值不在「哪个更好」，而在看清**谁在替你管循环、谁在替你管工具**。

| 写法 | 创建 Agent | 工具定义 | 谁跑工具循环 | 代码量 | 适用场景 |
|---|---|---|---|---|---|
| 原生 OpenAI SDK | 手写 `for` 循环 | 手写 JSON Schema | **你自己** | 约 80 行主循环 | 理解原理 / 需要完全掌控 |
| LangChain | `create_agent(...)` 一行 | `@tool` 装饰器（docstring + 类型注解自动转 Schema） | **框架**（LangGraph 图） | 一行创建 | 大多数业务：轻量、够用、可加中间件 |
| DeepAgents | `create_deep_agent(...)` 一行 | 与 LangChain **完全相同** | **框架**（同上） | 一行创建 | 复杂长程任务：白送文件系统 / 任务拆解 / 子 Agent / 上下文压缩 |

上面这张表是本节的地图，后面每一段代码都只是它在某一列上的展开。

> **本 notebook 由 `Agent/04_function_call/` 下 6 个脚本合并而成**：
> `agent_openai.py` + `agent_openai_jxsd.py`（路径 A 原生 SDK 手写循环）、
> `agent_langchain.py` + `agent_langchain_jxsd.py`（路径 B LangChain）、
> `agent_deepagents.py` + `agent_deepagents_jxsd.py`（路径 C DeepAgents）。
> 原脚依赖的同目录 `tools.py` / `tool_desc.py` 已归档到 `Agent/_py_source/`，
> 本节把它们需要的定义**内联**进来，notebook 因此可以独立跑，不依赖归档区的 import。

**官方文档**
- LangChain Agents：<https://docs.langchain.com/oss/python/langchain/agents>
- DeepAgents 总览：<https://docs.langchain.com/oss/python/deepagents/overview>
- OpenAI Function Calling：<https://developers.openai.com/api/docs/guides/function-calling>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型（本机 `deepseek-flash`） |
| 依赖 | `openai` / `langchain` / `langchain-core` / `deepagents`（本项目 venv 已装） |
| 密钥 | `settings.api_key`（仓库根 `.env` 已配置） |
| 前置服务 | 无（不起服务、不读数据库、不写磁盘） |
| 预计耗时 | 约 3~6 分钟（三个演示会真实往返多轮，最长的演示会转到循环上限） |

⚠️ **本机特有的一处降级，先交代清楚**：

演示 2 会传 `tool_choice="required"`（强制模型第一轮必须调工具），
而本机接的 `deepseek-flash` 是**思考模型，不支持强制工具选择**，
端点会返回 `400 Thinking mode does not support this tool_choice`。

源码里内置了「**打印说明 + 退回 `auto` 重试**」的降级分支，本节原样保留。
所以**在本机看到的是那条 `[降级]` 提示**（属于设计好的路径，不是报错）；
**换一个支持强制工具选择的端点（如 GPT-4o 系），同一条代码就能看到原始行为**：
第一轮被逼着调工具，主循环的往返轨迹因此稳定复现。

## 本节地图

```mermaid
graph LR
    U["用户问题<br/>2+4*6"] --> A["路径 A<br/>原生 OpenAI SDK"]
    U --> B["路径 B<br/>LangChain"]
    U --> C["路径 C<br/>DeepAgents"]
    A --> A1["你写的 for 循环<br/>解析 tool_calls → 执行 → 回填 tool 消息"]
    B --> B1["create_agent<br/>model 节点 ↔ tools 节点"]
    C --> C1["create_deep_agent<br/>= create_agent + 全套预装中间件"]
    A1 --> R["最终答案"]
    B1 --> R
    C1 --> R
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 路径 | 入口 | 循环在哪 | 工具描述从哪来 | 本节的完整版源文件 |
|---|---|---|---|---|
| A 原生 SDK | `client.chat.completions.create(...)` | **本 notebook 里那个 `for turn in range(max_turns)`** | 手写 JSON Schema（`tool_desc.list_tools()` 动态生成） | `agent_openai_jxsd.py` |
| B LangChain | `create_agent(model, tools, system_prompt)` | LangGraph 图内部（model 节点 ↔ tools 节点） | `@tool` 装饰器自动导出 | `agent_langchain_jxsd.py` |
| C DeepAgents | `create_deep_agent(model, tools, system_prompt)` | 同上（它就是 `create_agent` 加中间件） | 同上，**一个字都不用改** | `agent_deepagents_jxsd.py` |

**与上下节的衔接**：

- 上一节（`01_工具定义与参数类型`）讲「工具本体 + JSON Schema 怎么写」；
- 本节把这些工具**真的接给模型**，并用三种写法各跑一遍；
- 后面几节会顺着路径 C 展开 DeepAgents 的中间件（文件系统 / 子 Agent / HITL）。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 0.1 前置条件自检

这一格只做三件事，**不发任何网络请求**：

1. 逐个 `import` 本节要用的四个包，缺哪个就打印中文提示；
2. 检查 `settings.api_key` / `settings.base_url` 有没有读到；
3. 把结论写进 `READY` —— 后面每一格**真正调模型的代码都包在 `if READY:` 里**，
   条件不满足时它们会整体跳过并打印跳过提示，而不是抛一堆 Traceback。

> ⚠️ 本节**不需要磁盘写入**：路径 C 里看到的 `write_file` 写的是 DeepAgents 的
> **状态后端**（默认 `StateBackend`，存在图状态里），不会在仓库里建文件。

In [ ]:
# ---------- 前置条件自检：缺东西就打印中文提示，后续单元格整体跳过 ----------
import importlib

READY = True

for _pkg in ("openai", "langchain", "langchain_core", "deepagents"):
    try:
        importlib.import_module(_pkg)
    except ImportError as exc:                  # noqa: BLE001 —— 缺依赖只需要提示，不必中断
        READY = False
        print(f"[跳过] 缺少依赖 {_pkg}：{exc}（安装命令：uv add {_pkg}）")

from config import settings

if not settings.api_key:
    READY = False
    print("[跳过] 没读到 API Key：请在仓库根 .env 里配好 API_KEY / BASE_URL / MODEL_NAME")
if not settings.base_url:
    READY = False
    print("[跳过] 没读到 BASE_URL：请在仓库根 .env 里配好 API_KEY / BASE_URL / MODEL_NAME")

print(f"模型：{settings.model_name}    接口：{settings.base_url}")
print("前置条件自检：", "就绪" if READY else "条件不足（下面每一格都会打印 [跳过] 提示）")

### 预期输出

```text
模型：deepseek-flash    接口：https://api.deepseek.com
前置条件自检： 就绪
```

## 0.2 公共设施：把 `tools.py` / `tool_desc.py` 内联进来

三条路径用的是**同一批工具**，所以先把它们备齐。归档区里原本是两个文件：

| 归档文件 | 提供什么 | 谁在用 |
|---|---|---|
| `tools.py` | `get_weather` / `get_current_time` 两个普通函数 + `TOOL_REGISTRY` | 路径 A 的课案原版（`agent_openai.py`） |
| `tool_desc.py` | 这两把工具的 JSON Schema（`TOOLS`） | 同上 |
| `tools_jxsd.py` | 13 个工具函数（4 个数学 + 9 个参数类型示范） | 路径 A 的完整版 |
| `tool_desc_jxsd.py` | `list_tools()` / `call_tool()` / `TOOL_PARAMETERS` | 同上 |

本节把它们**内联成一个 notebook 内的定义集**，理由有两条：

1. 归档区（`Agent/_py_source/`）不在 `sys.path` 上，notebook 不该依赖它；
2. 三条路径共用同一批工具，工具定义出现一次就够 —— 这正是「工具定义在三个版本里不变」这句话的证据。

下面先把工具函数抄进来（**docstring 一字不改**：模型看到的就是它们）。

In [ ]:
import json
import random


# ---------- 天气 / 时间：路径 A 课案原版用的两个工具（来自归档的 tools.py）----------
def get_weather(city: str) -> str:
    """
    查询指定城市的天气。

    :param city: 城市名称，如「上海」
    :return: 天气描述字符串
    """
    weather_map = {
        "上海": ("晴", 25),
        "北京": ("多云", 18),
        "广州": ("阵雨", 30),
    }
    desc, temp = weather_map.get(city, ("未知", random.randint(0, 35)))
    return json.dumps({"city": city, "weather": desc, "temperature": temp},
                      ensure_ascii=False)


def get_current_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# ---------- 数学四则：课案《实例》原样（来自归档的 tools_jxsd.py）----------
def add_tool(a, b):
    """
    返回a+b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a+b的结果
    """
    return a + b


def sub_tool(a, b):
    """
    返回a-b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a-b的结果
    """
    return a - b


def mul_tool(a, b):
    """
    返回a*b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a*b的结果
    """
    return a * b


def div_tool(a, b):
    """
    返回a/b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a/b的结果
    """
    return a / b

### 0.2.1 课案《参数类型》小节的 9 个工具

这 9 个函数把 JSON Schema 的 8 种参数类型 + `enum` 各落成一个**真函数**，
路径 A 的演示 3 会拿它们考模型：「不同类型参数到底会不会被填对」。

| 类型 | 工具函数 |
|---|---|
| `string` | `format_username_tool` |
| `number` | `calc_price_tool` |
| `integer` | `check_stock_tool` |
| `boolean` | `toggle_feature_tool` |
| `array` | `summarize_tags_tool` |
| `object`（含嵌套） | `describe_user_tool` |
| `null`（`["string", "null"]`） | `set_nickname_tool` |
| `enum` | `convert_temperature_tool` |
| 多类型组合 | `create_user_tool` |

⚠️ 命名后缀是**约定**也是**机制**：`list_tools()` 靠 `dir()` + `endswith("_tool")`
扫工具，所以任何以 `_tool` 结尾的顶层函数都会被自动发给模型。

In [ ]:
def format_username_tool(username: str) -> str:
    """
    规范化用户名：去掉首尾空格、转小写、首字母大写
    :param username: 用户名（string 类型）
    :return: 规范化后的用户名
    """
    # 真实业务里「规范化」比「原样返回」更能看出参数确实传进来了
    return username.strip().lower().capitalize()


def calc_price_tool(price: float) -> str:
    """
    计算商品的含税价格（税率 13%）
    :param price: 商品价格（number 类型，整数或小数都可以）
    :return: 含税价格的文字说明
    """
    total = round(price * 1.13, 2)
    return f"价格 {price} 元，含税（13%）后 {total} 元"


def check_stock_tool(count: int) -> str:
    """
    检查库存是否充足
    :param count: 商品数量（integer 类型，只接受整数）
    :return: 库存状态说明
    """
    if count >= 10:
        return f"库存 {count} 件，充足"
    return f"库存 {count} 件，偏少，建议补货"


def toggle_feature_tool(is_enabled: bool) -> str:
    """
    开启或关闭某个功能开关
    :param is_enabled: 是否启用（boolean 类型，只能是 true / false）
    :return: 开关状态说明
    """
    return f"功能已{'开启' if is_enabled else '关闭'}"


def summarize_tags_tool(tags: list[str]) -> str:
    """
    统计标签：去重后按字典序输出
    :param tags: 标签列表（array 类型，元素是 string）
    :return: 标签统计结果
    """
    unique = sorted(set(tags))
    return f"共传入 {len(tags)} 个标签，去重后 {len(unique)} 个：{'、'.join(unique)}"


def describe_user_tool(user: dict) -> str:
    """
    把用户信息渲染成一句话（user 里可以带嵌套的 address 对象）
    :param user: 用户信息对象，形如 {"name": "张三", "age": 28,
                 "address": {"city": "南昌", "zip": "330000"}}
    :return: 用户描述
    """
    # 对象参数到了 Python 这边就是 dict，取值时务必用 .get 兜底：
    # 模型偶尔会漏字段，直接 user["age"] 会 KeyError 把主循环炸掉
    name = user.get("name", "匿名用户")
    age = user.get("age", "未知")
    address = user.get("address") or {}
    city = address.get("city", "未填写")
    zip_code = address.get("zip", "未填写")
    return f"{name}，{age} 岁，所在城市 {city}，邮编 {zip_code}"


def set_nickname_tool(optional_field: str | None = None) -> str:
    """
    设置昵称，允许传空（null）
    :param optional_field: 昵称，可以为 null（type 写成 ["string", "null"]）
    :return: 设置结果说明
    """
    if optional_field is None:
        return "昵称已清空（模型传入的是 null）"
    return f"昵称已设置为：{optional_field}"


def convert_temperature_tool(value: float, unit: str = "celsius") -> str:
    """
    温度单位换算（摄氏度 <-> 华氏度）
    :param value: 温度数值（number 类型）
    :param unit: 传入数值的单位，只能是 "celsius" 或 "fahrenheit"（enum 枚举，string 类型）
    :return: 换算结果
    """
    # enum 的作用就是**把取值限定在固定几个里**，模型只能从中挑一个；
    # 函数这边仍然要按普通字符串处理，不能假设模型一定守规矩
    if unit == "celsius":
        return f"{value}°C = {round(value * 9 / 5 + 32, 1)}°F"
    if unit == "fahrenheit":
        return f"{value}°F = {round((value - 32) * 5 / 9, 1)}°C"
    return f"不认识单位 {unit}，只支持 celsius / fahrenheit"


def create_user_tool(name: str, email: str, age: int = 0,
                     is_active: bool = True, tags: list[str] | None = None) -> str:
    """
    创建用户（对应课案《参数类型 → 8. 完整示例》里的 create_user）
    :param name: 用户名（string）
    :param email: 邮箱（string）
    :param age: 年龄（integer）
    :param is_active: 是否激活（boolean）
    :param tags: 标签（array，元素是 string）
    :return: 创建结果的 JSON 字符串
    """
    user = {
        "name": name,
        "email": email,
        "age": age,
        "is_active": is_active,
        "tags": tags or [],
    }
    # 返回 JSON 字符串而不是 dict：工具结果最终要拼进 messages 的 content 字段，
    # 那里只接受字符串
    return json.dumps({"已创建用户": user}, ensure_ascii=False)

### 0.2.2 注册表、JSON Schema 与「按名字分发」

这一格把三样东西备齐，它们对应归档里的 `tools.py` / `tools_jxsd.py` / `tool_desc.py` / `tool_desc_jxsd.py`：

1. **`TOOL_REGISTRY`** —— 模型报出函数名后，程序靠它找到真函数；
   白名单式注册表还有安全作用：模型只能调到这里列出的函数。
2. **`TOOL_PARAMETERS`** —— 每个工具的参数 JSON Schema。
   课案里 `parameters` 是写死的 `{a: number, b: number}`（只够 4 个数学工具用），
   这里改成按工具名查表，工具扩容到 13 个也不用改主循环。
3. **`TOOLS`（天气/时间的 2 个声明）** 与 **`list_tools()` / `call_tool()`**。

> **为什么要造两个「模块对象」**：下面路径 A 的源码里写的是
> `from tools import TOOL_REGISTRY` / `import tools_jxsd`（归档时是两个真文件）。
> 归档区不在 `sys.path` 上，本 notebook 也不该依赖它 ——
> 于是这里把上面内联好的定义装配成**真正的模块对象**注册进 `sys.modules`。
> 这样那几行 `import` 可以**照原样保留、照原样生效**，而 notebook 仍然自包含。
>
> ⚠️ 装配时踩到过一个真坑：`list_tools()` 是靠 `endswith("_tool")` 扫模块的，
> 而本节还有一个辅助函数 `call_tool` 也以 `_tool` 结尾 —— 照「所有 `*_tool` 名字」
> 装模块，它就会被当成**第 14 个工具**发给模型。所以下面显式只装 13 个真工具函数。

In [ ]:
# ---------- 工具注册表 ----------
# 数学四则工具：课案的「实例」只用这 4 个
MATH_TOOL_NAMES = ["add_tool", "sub_tool", "mul_tool", "div_tool"]

# 参数类型示范工具：课案《参数类型》小节那 8 种类型 + enum
PARAM_TOOL_NAMES = [
    "format_username_tool",
    "calc_price_tool",
    "check_stock_tool",
    "toggle_feature_tool",
    "summarize_tags_tool",
    "describe_user_tool",
    "set_nickname_tool",
    "convert_temperature_tool",
    "create_user_tool",
]

TOOL_REGISTRY = {
    "add_tool": add_tool,
    "sub_tool": sub_tool,
    "mul_tool": mul_tool,
    "div_tool": div_tool,
    "format_username_tool": format_username_tool,
    "calc_price_tool": calc_price_tool,
    "check_stock_tool": check_stock_tool,
    "toggle_feature_tool": toggle_feature_tool,
    "summarize_tags_tool": summarize_tags_tool,
    "describe_user_tool": describe_user_tool,
    "set_nickname_tool": set_nickname_tool,
    "convert_temperature_tool": convert_temperature_tool,
    "create_user_tool": create_user_tool,
}
# 路径 A 的课案原版（agent_openai.py）用它查天气/时间工具，所以也登记进来
TOOL_REGISTRY["get_weather"] = get_weather
TOOL_REGISTRY["get_current_time"] = get_current_time

In [ ]:
# ---------- 天气 / 时间这两个工具的 JSON Schema（归档 tool_desc.py 的 TOOLS）----------
GET_WEATHER_DESC = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询指定城市的实时天气，包括天气状况和温度",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，例如：上海",
                },
            },
            "required": ["city"],
        },
    },
}

GET_TIME_DESC = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "获取当前的日期和时间",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    },
}

# 发给 API 的 tools 参数：工具描述列表
TOOLS = [GET_WEATHER_DESC, GET_TIME_DESC]

In [ ]:
# ---------- 每个 `_tool` 函数的参数 Schema（归档 tool_desc_jxsd.py 的 TOOL_PARAMETERS）----------
# 没有描述的工具走这个兜底 Schema：一个不接任何参数的 object
EMPTY_PARAMETERS = {"type": "object", "properties": {}, "required": [], "additionalProperties": False}

# 课案 agent.py 给 4 个数学工具写死的参数：a、b 都是 number
SCHEMA_MATH_A = {"name": "a", "type": "number", "description": "第一个数字"}
SCHEMA_MATH_B = {"name": "b", "type": "number", "description": "第二个数字"}

TOOL_PARAMETERS = {
    # 数学四则：对应课案 agent.py 里写死的 {"a": number, "b": number}
    **{name: {"type": "object",
              "properties": {"a": {"type": "number", "description": "第一个数字"},
                             "b": {"type": "number", "description": "第二个数字"}},
              "required": ["a", "b"],
              "additionalProperties": False}
       for name in MATH_TOOL_NAMES},
    # 课案《参数类型》8 种类型，逐个落到上面那些真函数上
    "format_username_tool": {"type": "object",
                             "properties": {"username": {"type": "string", "description": "用户名"}},
                             "required": ["username"], "additionalProperties": False},
    "calc_price_tool": {"type": "object",
                        "properties": {"price": {"type": "number", "description": "商品价格"}},
                        "required": ["price"], "additionalProperties": False},
    "check_stock_tool": {"type": "object",
                         "properties": {"count": {"type": "integer", "description": "商品数量"}},
                         "required": ["count"], "additionalProperties": False},
    "toggle_feature_tool": {"type": "object",
                            "properties": {"is_enabled": {"type": "boolean", "description": "是否启用"}},
                            "required": ["is_enabled"], "additionalProperties": False},
    "summarize_tags_tool": {"type": "object",
                            "properties": {"tags": {"type": "array", "items": {"type": "string"},
                                                    "description": "标签列表"}},
                            "required": ["tags"], "additionalProperties": False},
    # 对象 + 嵌套对象（对象里再套一个对象），required 覆盖全部字段
    "describe_user_tool": {
        "type": "object",
        "properties": {
            "user": {
                "type": "object",
                "description": "用户信息对象",
                "properties": {
                    "name": {"type": "string", "description": "用户名"},
                    "age": {"type": "integer", "description": "年龄"},
                    "address": {
                        "type": "object",
                        "description": "地址（对象里再套一个对象）",
                        "properties": {"city": {"type": "string", "description": "城市"},
                                       "zip": {"type": "string", "description": "邮编"}},
                        "required": ["city", "zip"],
                        "additionalProperties": False,
                    },
                },
                "required": ["name", "age", "address"],
                "additionalProperties": False,
            },
        },
        "required": ["user"],
        "additionalProperties": False,
    },
    "set_nickname_tool": {"type": "object",
                          "properties": {"optional_field": {"type": ["string", "null"],
                                                            "description": "可选字段"}},
                          "required": ["optional_field"], "additionalProperties": False},
    # enum：把取值限定在固定几个里（注意 type 是 string，不是课案笔误里的 number）
    "convert_temperature_tool": {"type": "object",
                                 "properties": {"value": {"type": "number", "description": "温度数值"},
                                                "unit": {"type": "string",
                                                         "enum": ["celsius", "fahrenheit"],
                                                         "description": "传入数值的单位"}},
                                 "required": ["value", "unit"], "additionalProperties": False},
    # 课案《8. 完整示例》的 create_user：多类型组合
    "create_user_tool": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "用户名"},
            "age": {"type": "integer", "description": "年龄"},
            "email": {"type": "string", "description": "邮箱"},
            "is_active": {"type": "boolean", "description": "是否激活"},
            "tags": {"type": "array", "items": {"type": "string"}, "description": "标签"},
        },
        "required": ["name", "email"],
        "additionalProperties": False,
    },
}

In [ ]:
# ---------- list_tools / call_tool（归档 tool_desc_jxsd.py 的两个函数）----------
def list_tools(only: list[str] | None = None) -> list[dict]:
    """
    列出 tools_jxsd 里所有的工具

    :param only: 可选，只列这几个工具名；不传 = 列出全部（课案原行为）
    :return: [{"工具名": ..., "工具描述": ..., "参数": ...}, ...]

    「工具名 / 工具描述 / 参数」这几个中文键名是课案的习惯，只在本项目内部用，
    发给模型前一定要映射成 name / description / parameters（见下面的 tools 声明）。
    """
    names = [func for func in dir(tools_jxsd) if func.endswith("_tool")]
    if only is not None:
        # 排序保证顺序稳定：dir() 返回的列表是按字母序的，过滤后仍然是字母序
        names = [func for func in names if func in only]
    return [
        {
            "工具名": name,
            "工具描述": getattr(tools_jxsd, name).__doc__,
            "参数": TOOL_PARAMETERS.get(name, EMPTY_PARAMETERS),
        }
        for name in names
    ]


def call_tool(tool_name, *args, **kwargs):
    """
    调用工具：按名字从注册表里取出真函数并执行

    参数是**模型生成的、不可信**，所以这里有两层保护：
        1. 名字白名单：模型可能报出一个不存在的函数名（getattr 会抛 AttributeError）；
        2. 执行兜底：模型可能给出 div_tool(a=1, b=0) 这种非法参数。
    这两类错误都不该让主循环崩掉 —— 把错误当成「工具执行结果」回传给模型，
    模型下一轮往往能自己纠正（这正是 Agent 自愈能力的来源）。
    """
    func = TOOL_REGISTRY.get(tool_name)
    if func is None:
        return f"没有名为 {tool_name} 的工具"
    try:
        return func(*args, **kwargs)
    except Exception as exc:                      # noqa: BLE001 —— 故意兜住全部异常，回传给模型
        return f"工具 {tool_name} 执行出错：{type(exc).__name__}: {exc}"

In [ ]:
# ---------- 把内联定义装配成「与归档文件同名」的模块对象 ----------
# 归档里 tools.py / tools_jxsd.py 是一件事的两种版本（课案原版 / 完整版），
# tool_desc.py / tool_desc_jxsd.py 同理，所以这里各装配一个模块、注册两个名字。
import types

_tools_module = types.ModuleType("tools_jxsd")
# ⚠️ 这里**不能**用「当前命名空间里所有以 `_tool` 结尾的名字」来装模块：
# 本节还有一个辅助函数 `call_tool` 也以 `_tool` 结尾，一起装进去就会被
# list_tools() 当成第 14 个工具发给模型（实测过，模型真的会看到它）。
# 所以只装 MATH_TOOL_NAMES + PARAM_TOOL_NAMES 这 13 个真正的工具函数。
_TOOL_FUNCTION_NAMES = MATH_TOOL_NAMES + PARAM_TOOL_NAMES
_tools_module.__dict__.update({_name: globals()[_name] for _name in _TOOL_FUNCTION_NAMES})
_tools_module.MATH_TOOL_NAMES = MATH_TOOL_NAMES
_tools_module.PARAM_TOOL_NAMES = PARAM_TOOL_NAMES
_tools_module.TOOL_REGISTRY = TOOL_REGISTRY
sys.modules["tools"] = _tools_module
sys.modules["tools_jxsd"] = _tools_module
tools_jxsd = _tools_module                      # 让后面「照源文件写」的代码能直接用这个模块名

_tool_desc_module = types.ModuleType("tool_desc_jxsd")
_tool_desc_module.list_tools = list_tools
_tool_desc_module.call_tool = call_tool
_tool_desc_module.TOOLS = TOOLS
_tool_desc_module.TOOL_PARAMETERS = TOOL_PARAMETERS
sys.modules["tool_desc"] = _tool_desc_module
sys.modules["tool_desc_jxsd"] = _tool_desc_module
tool_desc = _tool_desc_module

print(f"内联装配完成：tools/tools_jxsd 共 {len(TOOL_REGISTRY)} 个已登记工具，"
      f"tool_desc/tool_desc_jxsd 的 TOOLS 有 {len(TOOLS)} 个声明")

### 预期输出

```text
内联装配完成：tools/tools_jxsd 共 15 个已登记工具，tool_desc/tool_desc_jxsd 的 TOOLS 有 2 个声明
```

**15 = 13 个 `_tool` 函数 + 天气 + 时间**；而列表名说 `TOOLS` 只有 2 个 ——
那是因为 `TOOLS` 就是归档 `tool_desc.py` 里那两条天气/时间声明，
路径 A 完整版的 13 个工具声明是后面用 `list_tools()` 动态生成的。

## 1. 课案原版：三条路径的最短实现（34 + 28 + 28 = 90 行）

先看三条路径**各自最短的样子**。三格代码做的事完全一样：
让模型自己决定调不调工具、调哪个，然后把答案说出来。

| | 路径 A `agent_openai.py` | 路径 B `agent_langchain.py` | 路径 C `agent_deepagents.py` |
|---|---|---|---|
| 有效代码行 | 34 | 28 | 28 |
| 循环 | **手写 `for _ in range(10)`** | `create_agent` 内部 | `create_deep_agent` 内部 |
| 工具描述 | 手写 JSON（`tool_desc.TOOLS`） | `@tool` 自动导出 | `@tool` 自动导出 |
| 工具结果回填 | 手写 `{"role": "tool", "tool_call_id": ...}` | 框架自动 | 框架自动 |
| 额外能力 | 无 | 无 | 文件系统 / 任务拆解 / 子 Agent / 上下文压缩 |

注意 A 与 B/C 的**语言差异**：A 用的是 `openai` SDK 的原始消息字典，
B/C 用的是 LangChain 的 `@tool` 对象；但 B 和 C 之间**工具定义一个字都不用改**。

### 1.1 路径 A · 原生 OpenAI SDK：手写完整 Function Call 循环

这是本章最重要的一段：**不借助任何框架**，把 Function Call 的回路手写一遍。

```mermaid
sequenceDiagram
    participant P as 程序
    participant M as 模型
    participant T as 工具函数
    P->>M: system + user + tools 声明
    M->>P: message.tool_calls = [get_weather(city="上海")]
    P->>T: TOOL_REGISTRY["get_weather"](city="上海")
    T->>P: '{"city": "上海", "weather": "晴", ...}'
    P->>M: {"role": "tool", "tool_call_id": ..., "content": ...}
    M->>P: 最终自然语言回答（不再返回 tool_calls）
```

四个容易踩的坑（课案《关键点说明》都点到了）：

1. `tool_call_id` 必须一一对应，工具结果靠 id 认领；
2. assistant 那轮必须把 `tool_calls` 原样存回历史，否则模型不知道刚才调过什么；
3. 必须设循环上限（这里 `range(10)`），否则模型来回调同一个工具就是死循环 + 一直烧 token；
4. `role="tool"` 的 `content` 只能是字符串，工具返回 dict/数字都要 `str()` 一下。

In [ ]:
from openai import OpenAI
from config import settings

from tools import TOOL_REGISTRY
from tool_desc import TOOLS

# OpenAI 兼容客户端（DeepSeek / 通义 / vLLM 都能这样接）
client = OpenAI(api_key=settings.api_key, base_url=settings.base_url)

messages = [
    {"role": "system", "content": "你是一个生活助手，查询天气和时间请使用工具。"},
    {"role": "user", "content": "上海现在天气怎么样？现在几点了？"},
]

if READY:
    # ---------- Function Call 主循环 ----------
    for _ in range(10):  # 防御：限制最多 10 轮，避免死循环
        response = client.chat.completions.create(
            model=settings.model_name,
            messages=messages,
            tools=TOOLS,             # 把工具描述发给模型
            tool_choice="auto",      # 模型自己决定是否调工具
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump())

        # 模型没有要求调工具 → 说明已有最终答案，退出循环
        if not msg.tool_calls:
            print("AI 最终回答：", msg.content)
            break

        # 执行模型要求的每一个工具调用
        for call in msg.tool_calls:
            name = call.function.name
            import json
            args = json.loads(call.function.arguments)  # 参数是 JSON 字符串
            print(f"[模型请求调用] {name}({args})")

            # 从注册表找到真正的 Python 函数并执行
            result = TOOL_REGISTRY[name](**args)
            print(f"[工具执行结果] {result}")

            # 工具结果回传给模型（tool_call_id 必须对应）
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": result,
            })
else:
    print("[跳过] 前置条件不足，路径 A 的课案原版演示未执行")

### 预期输出

```text
[模型请求调用] get_weather({'city': '上海'})
[工具执行结果] {"city": "上海", "weather": "晴", "temperature": 25}
[模型请求调用] get_current_time({})
[工具执行结果] 2026-09-17 15:38:10
AI 最终回答： 以下是查询结果：

🌤️ **上海天气**
- 天气状况：晴
- 温度：25°C

🕒 **当前时间**
- 2026年9月17日 15:38:10

上海现在天气晴朗，气温25°C，非常舒适。不过这个时间紫外线可能仍较强，外出记得做好防晒措施哦！如需了解其他信息，随时告诉我。
```

**这一格正好把 Function Call 的两次往返都跑出来了**：

1. 第一轮模型一次要了**两个**工具（`get_weather` + `get_current_time`）——
   注意原始 SDK 的 `call.function.arguments` 是 **JSON 字符串**，所以要 `json.loads`；
2. 工具结果按 `tool_call_id` 回填后，模型才输出最终自然语言答案。

> ⚠️ 模型措辞**每次运行都不同**，时间戳也是当次运行的真实值：
> 上面的正文只是我这次跑出来的**实测值**，**别逐字比对**。
> 不变的是结构：`[模型请求调用] …` → `[工具执行结果] …` → `AI 最终回答： …`。

### 1.2 路径 B · LangChain：`create_agent` 一行

对比上一格那 80 行手写循环，这里只差 `create_agent(model, tools, system_prompt)` 一行 ——
工具循环、历史记录、错误处理它全包了。

工具定义也从「函数 + 手写 JSON Schema」简化为一个 `@tool` 装饰器：

| `@tool` 里的东西 | 变成 JSON 里的 |
|---|---|
| docstring | `function.description` |
| 类型注解 + 形参名 | `function.parameters`（properties / required） |
| 函数名 | `function.name` |

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ---------- 用 @tool 定义工具：描述自动生成，无需手写 JSON Schema ----------
@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气，包括天气状况和温度"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度", "广州": "阵雨 30 度"}
    return weather_map.get(city, f"{city} 天气未知")


@tool
def get_current_time() -> str:
    """获取当前的日期和时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


agent = create_agent(
    model=llm,
    tools=[get_weather, get_current_time],
    system_prompt="你是一个生活助手，查询天气和时间请使用工具。",
)

if READY:
    result = agent.invoke(
        {"messages": [("user", "上海现在天气怎么样？现在几点了？")]}
    )
    print("AI 最终回答：", result["messages"][-1].content)
else:
    print("[跳过] 前置条件不足，路径 B 的课案原版演示未执行")

### 预期输出

```text
AI 最终回答： 上海现在天气晴，气温 25 度；当前时间是 2026 年 9 月 17 日 15:38:14。

天气不错，25 度晴天挺舒服的，适合外出活动。☀️
```

一样的工具、一样的问法，路径 B 只用一行 `create_agent` 就把这两次工具往返跑完了。
想看它内部**怎么转**的，得把 `result["messages"]` 翻出来 —— 3.2 节的 `print_trace` 就是干这个的。

> ⚠️ 上面那段正文是**模型自己的措辞**，里面还嵌着**当次运行的时间戳**，
> 所以它**每次运行都不同**：这只是我这次跑出来的**实测值**，**别逐字比对**。
> 稳定不变的是结构 —— 「通过工具查到天气与时间，再用自然语言汇总」这件事。

### 1.3 路径 C · DeepAgents：`create_deep_agent` 一行

课案原话：「DeepAgents 封装了 LangChain 的 `create_agent`，内置文件系统、
任务拆解等中间件。对工具调用来说，**API 完全一致**，只是换成 `create_deep_agent`」。

所以这一格和上一格的差别只有两处：

- 函数名 `create_agent` → `create_deep_agent`；
- `system_prompt` 里多要求了一句「先列任务清单 / 整理成表格文件保存」——
  这正是 DeepAgents 白送的能力（**我们并没有定义任何文件工具**）。

In [ ]:
from deepagents import create_deep_agent

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气，包括天气状况和温度"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度", "广州": "阵雨 30 度"}
    return weather_map.get(city, f"{city} 天气未知")


agent = create_deep_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        "你是生活助手。多城市查询时先列任务清单（todo），"
        "把结果整理成表格文件保存，最后汇报。"
    ),
)

if READY:
    result = agent.invoke(
        {"messages": [("user", "对比一下上海、北京、广州三地天气，整理成表格")]},
        config={"recursion_limit": 50},
    )
    print("AI 最终回答：", result["messages"][-1].content)
else:
    print("[跳过] 前置条件不足，路径 C 的课案原版演示未执行")

### 预期输出

```text
AI 最终回答： ✅ 三地天气对比已完成，结果已保存至 `/weather_comparison.md`。

**天气对比表：**

| 城市 | 天气状况 | 温度 |
| --- | --- | --- |
| 上海 | 晴 ☀️ | 25 度 |
| 北京 | 多云 ⛅ | 18 度 |
| 广州 | 阵雨 🌧️ | 30 度 |

**小结：**
- 🌡️ **气温**：广州最热（30 度）> 上海（25 度）> 北京最凉（18 度），三地最大温差 12 度。
- ☁️ **天气**：只有上海是晴天，北京多云，广州有阵雨。
- 💡 **建议**：广州记得带伞；北京偏凉注意保暖；上海天气最好，适合出门。

需要我把表格导出成 Excel（.xlsx）或 CSV 格式吗？
```

**注意这里没有任何一行代码是我们写的工具**：`write_todos`（列任务清单）和
`write_file`（存表格文件）都是 `create_deep_agent` 预装中间件带来的 ——
换成路径 B 的 `create_agent`，模型手里根本没有文件工具，这句话它做不到。

> 同一次运行里，模型也可能只回一段正文而不真去调 `write_file`（模型偏好问题），
> 那是模型的自由裁量，不是配置问题；4.3 节的演示 2 用「把工具名点出来」的方式把它稳定下来。
>
> ⚠️ 上面那段正文**完全由模型决定**：措辞每次不同，表格的长宽、有没有额外建议也每次不同，
> 文件名甚至可能是别的路径 —— 这只是我这次跑出来的**实测值**，**别逐字比对**。
> 稳定不变的是三件事：① 三地天气都被查了；② 表格被写进了文件（`write_file` 真的执行了）；
> ③ 最后给了一段汇总。这三件事才是这一格要看的证据。

## 2. 完整版 · 路径 A：手写 Function Call 主循环

课案原版那 34 行是「最短实现」，能跑通但没有防御、没有计数器、看不到循环转了几圈。
完整版把它补成一个可以复用的函数，并加了**三个演示**：

| 演示 | 用户问题 | 首轮 `tool_choice` | 想说明什么 |
|---|---|---|---|
| 演示 1 | `2+4*6`（带 Few-shot 历史） | `auto` | 课案《实例》原样；也能看到「Few-shot 对模型是引导还是干扰」因模型而异 |
| 演示 2 | `2+4*6`（去掉 Few-shot） | **`required`** | 稳定复现课案的两步轨迹；**本机在这里走降级分支** |
| 演示 3 | 建用户 + 温度换算 | `auto` | `string` / `integer` / `boolean` / `array` / `enum` 参数真被填对了 |

### 2.1 客户端与工具声明：动态生成 tools

课案《关键点说明 1》：「动态生成 tools —— 通过列表推导式从 `tool_desc.list_tools()` 动态生成工具定义」。
好处是新增工具时只要加一个 `xxx_tool` 函数，这里**一个字都不用改**。

课案原文那处语法笔误（`parameters` 那行结尾少了 `}` 后的逗号，导致 `"strict": True`
被写进了 `parameters` 字典内部）在这里已按正确结构修正。

In [ ]:
import tools_jxsd                      # 真正的 Python 工具函数
import tool_desc_jxsd as tool_desc     # 工具描述；课案里这个模块叫 tool_desc

# 课案原句，一字未改
system_prompt = "你是一个助手，可以帮助用户进行数学计算。"

tools = [
    {
        "type": "function",
        "function": {
            "name": tool["工具名"],
            "description": tool["工具描述"],
            "parameters": tool["参数"],
            "strict": True,      # 与 additionalProperties: False 配对使用
        },
    }
    for tool in tool_desc.list_tools()
]

print(f"模块顶部 tools 共 {len(tools)} 个（= list_tools() 扫到的全部工具）")

### 预期输出

```text
模块顶部 tools 共 13 个（= list_tools() 扫到的全部工具）
```

⚠️ `strict: True` 有硬性配套条件：每个 object 都要写 `additionalProperties: False`，
且 `required` 必须覆盖 `properties` 的全部字段 —— 少一条服务端就返回 400，
而报错信息通常只说 schema 不合法，不告诉你是哪一条。

### 2.2 Few-shot 历史：在历史对话里示范「怎么调工具」

课案《关键点说明 2》：「Few-shot 示例 —— 在历史对话中提供示例，帮助模型理解如何调用工具」。
这段历史本身就是一本工具调用示范教材，它在教模型三件事：

1. 复杂算式要**拆成多步**，一步只调一个工具；
2. 每轮 assistant 消息里要带 `tool_calls`，工具结果用 `role="tool"` 回填；
3. 最终答案要基于工具结果算出来（`16-9=7` → `8*2-9=7`）。

⚠️ 注意 `arguments` 是 **JSON 字符串**，不是 dict —— 这是 OpenAI 协议的硬性规定，
所以下面用 `json.dumps` 造示例。

In [ ]:
def build_few_shot_history() -> list[dict]:
    """课案的 Few-shot 历史对话（8*2-9 的三段往返）"""
    return [
        {"role": "user", "content": "8*2-9"},
        # 第一轮：模型决定先算 8*2。注意 tool_calls 里的 arguments 是**JSON 字符串**，
        # 不是 dict —— 这是 OpenAI 协议的硬性规定，所以下面用 json.dumps 造示例
        {"role": "assistant", "content": "要计算8*2-9。首先计算8*2。", "tool_calls": [
            {"id": "call_1", "type": "function",
             "function": {"name": "mul_tool", "arguments": json.dumps({"a": 8, "b": 2})}}
        ]},
        # 工具结果回填：tool_call_id 必须等于上面那条的 "call_1"
        {"role": "tool", "tool_call_id": "call_1", "content": "16"},
        # 第二轮：拿到 16 之后，模型继续算 16-9
        {"role": "assistant", "content": "8*2=16，现在计算16-9。", "tool_calls": [
            {"id": "call_2", "type": "function",
             "function": {"name": "sub_tool", "arguments": json.dumps({"a": 16, "b": 9})}}
        ]},
        {"role": "tool", "tool_call_id": "call_2", "content": "7"},
        # 工具用完了，这一轮不再返回 tool_calls，而是直接给最终答案
        {"role": "assistant", "content": "16-9=7，所以8*2-9=7"},
    ]

### 2.3 主循环（含 `tool_choice` 降级分支）

循环体就是课案《实例》那段完整 Agent 循环，包成函数以便复用。三处值得单独说：

1. **每次都要把完整历史重新发一遍** —— Chat Completions API 是**无状态**的，
   服务端不记得你上一轮说了什么；
2. **首轮可以传 `required`**（强制模型第一轮必须调工具），
   但**绝不能每轮都用**：那样模型永远没机会输出最终答案，会一路撞到 `max_turns` ——
   这也正说明「循环上限」不是可有可无的摆设；
3. **降级分支**：端点拒绝强制工具选择时，打印说明并退回 `auto` 重试。
   实测本机 `deepseek-flash` 会返回 `400 Thinking mode does not support this tool_choice`。
   语义从「强迫模型必须调工具」变成「模型自己决定」，属于真的换了行为，
   所以**必须打印出来，不能静默处理**。

In [ ]:
def run_agent_loop(history: list[dict], tools: list[dict],
                   system_prompt: str = system_prompt, max_turns: int = 10,
                   first_tool_choice: str = "auto"):
    """
    执行 Function Call 主循环

    :param history: 对话历史（会被就地修改：追加 assistant 的 tool_calls 与 tool 结果）
    :param tools: 工具描述列表（发给模型的 tools 参数）
    :param system_prompt: system 提示词
    :param max_turns: 循环上限，课案原文是写死的 range(10)
    :param first_tool_choice: 首轮的 tool_choice；课案是 auto，
            传 "required" 可强制模型第一轮必须调工具
    :return: 模型的最终回答；达到上限仍未结束时返回 None
    """
    final_answer = None
    tool_calls_made = 0        # 本文件加的计数器：直观看出主循环到底转了几圈、调了几次工具
    turns_used = 0

    for turn in range(max_turns):
        turns_used = turn + 1

        tool_choice = first_tool_choice if turn == 0 else "auto"

        # 抽成内部函数：端点拒绝强制 tool_choice 时要原样重发一次（见下面的降级分支）
        def _create(choice: str):
            return client.chat.completions.create(
                model=settings.model_name,     # 课案原文是 setting.MODEL_NAME（settings 的另一种大小写风格）
                messages=[{"role": "system", "content": system_prompt}, *history],
                tools=tools,
                tool_choice=choice,
                temperature=0,     # 温度 0：算数题要的是稳定复现，不要随机发挥
            )

        try:
            response = _create(tool_choice)
        except Exception as exc:
            # 端点不支持强制工具选择时的降级。实测：**思考模型**（DeepSeek 的
            # deepseek-flash / deepseek-v4-pro）会返回
            #     400 Thinking mode does not support this tool_choice
            # 退回 auto 重试 —— 演示语义从「强迫模型必须调工具」变成「模型自己决定」，
            # 属于真的换了行为，所以必须打印出来，不能静默处理。
            if tool_choice == "auto":
                raise          # 本来就用 auto 还失败，那是别的问题，如实抛出去
            print(f"  [降级] 本端点不支持 tool_choice={tool_choice!r}"
                  f"（{type(exc).__name__}: {str(exc)[:90]}）")
            print("         改用 tool_choice='auto' 重试：本轮是否调工具由模型自行决定。")
            tool_choice = "auto"
            response = _create(tool_choice)

        message = response.choices[0].message

        # 如果没有工具调用，说明任务结束
        if not message.tool_calls:
            print("结束")
            print(f"最终答案: {message.content}")
            final_answer = message.content
            break

        # 记录assistant的tool_calls
        # 课案用 [dict(tc) for tc in ...] 把 pydantic 对象转成普通 dict 再入历史，
        # 而不是 message.model_dump()：model_dump 会带上 refusal / annotations
        # 等一堆空字段，部分兼容服务端见到陌生字段会直接报错
        history.append({
            "role": "assistant",
            "content": message.content,
            "tool_calls": [dict(tc) for tc in message.tool_calls],
        })

        # 处理每个工具调用（模型一轮可能同时要调好几个工具，所以是 for 不是 if）
        for tool_call in message.tool_calls:
            tool_calls_made += 1
            # arguments 是模型生成的 JSON **字符串**，必须自己 json.loads 成 dict
            args = json.loads(tool_call.function.arguments)
            # 按函数名分发到真实函数执行（课案：tool_desc.call_tool）
            result = tool_desc.call_tool(tool_call.function.name, **args)
            # 工具结果必须带 tool_call_id 回到历史里，否则模型对不上号
            history.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})
            # str(result)：role="tool" 的 content 只接受字符串，返回 dict/数字都要转一下
            print(f"调用工具: {tool_call.function.name}, 参数: {args}, 结果: {result}")
    else:
        # 课案原文没有这个分支（for-else 在没 break 时触发）；
        # 加上是为了让「防御上限真的被触发」这件事可见，而不是静默退出
        print(f"⚠️ 已循环 {max_turns} 次仍未得到最终答案，强制结束（防御上限生效）")

    print(f"[主循环共转 {turns_used} 圈，执行 {tool_calls_made} 次工具调用]")
    return final_answer


def _to_openai_tools(tool_list: list[dict]) -> list[dict]:
    """
    把 list_tools() 的结果转成 API 需要的 tools 参数

    抽成函数是因为模块顶部那份 tools 是「全量工具」，而各演示只想给一部分工具；
    转换逻辑（就是课案那段列表推导式）只应存在一份。
    """
    return [
        {
            "type": "function",
            "function": {
                "name": tool["工具名"],
                "description": tool["工具描述"],
                "parameters": tool["参数"],
                "strict": True,
            },
        }
        for tool in tool_list
    ]

### 2.4 三个演示

- **演示 1** 用 `only=tools_jxsd.MATH_TOOL_NAMES` 只给 4 个数学工具 ——
  课案当时的 `tools.py` 里就只有这 4 个，工具给多了模型反而分心；
- **演示 2** 与演示 1 只有两处差别：历史里没有那 6 条示例消息、首轮传 `required`；
- **演示 3** 换成 9 个参数类型工具，考的是「不同类型参数会不会被填对」。

In [ ]:
def demo_math_few_shot():
    """演示 1：课案《实例》原样 —— Few-shot 历史 + 数学题 2+4*6"""
    print("=" * 62)
    print("演示 1：课案《实例》原样（Few-shot 历史 + 用户问 2+4*6）")
    print("=" * 62)
    # 课案原文：
    #     user_prompt = "2+4*6"
    #     history.append({"role": "user", "content": user_prompt})
    history = build_few_shot_history()
    user_prompt = "2+4*6"
    history.append({"role": "user", "content": user_prompt})

    # 只给 4 个数学工具：课案当时的 tools.py 里就只有这 4 个，
    # list_tools(only=...) 正是为了还原这个场景（否则 13 个工具一起发，
    # 模型面对「创建用户」「查库存」这些无关工具反而更容易分心）
    math_tools = tool_desc.list_tools(only=tools_jxsd.MATH_TOOL_NAMES)
    print(f"本轮可用工具：{[t['工具名'] for t in math_tools]}")
    print()
    # 课案《运行结果示例》给出的预期输出：
    #     调用工具: mul_tool, 参数: {'a': 4, 'b': 6}, 结果: 24
    #     调用工具: add_tool, 参数: {'a': 2, 'b': 24}, 结果: 26
    #     结束
    #     最终答案: 2+4*6=26
    #
    # ⚠️ 本机实测提醒（详见演示 2）：把这段 Few-shot 一起喂给当前模型时，
    #    模型会**照着示例反复调 mul_tool**，甚至干脆不调工具直接作答——
    #    Few-shot 示例对它是干扰而非引导。这不是代码错，而是
    #    「提示词效果依模型而异」的真实体感，换 GPT-4o / DeepSeek 一类模型
    #    通常就能如期生效。
    return run_agent_loop(history, _to_openai_tools(math_tools), system_prompt)


def demo_math_plain():
    """演示 2：同一道题、同一 system 提示，去掉 Few-shot + 首轮强制调工具"""
    print()
    print("=" * 62)
    print("演示 2：去掉 Few-shot，首轮 tool_choice='required'（稳定复现课案轨迹）")
    print("=" * 62)
    # 与演示 1 的差别有两处：
    #   ① history 里没有那 6 条示例消息；
    #   ② 首轮 tool_choice 传 "required"（强制模型第一轮必须调工具）。
    #
    # 为什么要加 ②？实测当前模型在 auto 下相当随性：
    #   - 有时乖乖调 mul_tool，再调 add_tool（就是课案预期的那条轨迹）；
    #   - 有时直接心算给出 26，一个工具都不调；
    #   - 有时在正文里复述「调用 mul_tool with a=4, b=6」，却根本不发 tool_calls。
    # 首轮 required 至少能保证「确实调过工具」，让主循环真的转起来。
    #
    # ⚠️ 本机注意：deepseek-flash 是思考模型，**不支持强制 tool_choice**，
    #    这一格会走降级分支（打印 [降级] 后退回 auto）。换支持强制工具选择的
    #    端点，就能看到课案那条不降级的原始轨迹。
    #
    # 但要如实说明：第二轮仍是 auto，模型可能接着调 add_tool（课案预期），
    # 也可能自己心算把 2+24 算完就收工。这一段属于模型的自由裁量，脚本管不着。
    history = [{"role": "user", "content": "2+4*6"}]
    math_tools = tool_desc.list_tools(only=tools_jxsd.MATH_TOOL_NAMES)
    return run_agent_loop(history, _to_openai_tools(math_tools), system_prompt,
                          first_tool_choice="required")


def demo_param_types():
    """演示 3：课案《参数类型》小节的 9 个工具，看模型怎么填不同类型的参数"""
    print()
    print("=" * 62)
    print("演示 3：参数类型（string / integer / boolean / array / enum 真的被填对了）")
    print("=" * 62)
    param_tools = tool_desc.list_tools(only=tools_jxsd.PARAM_TOOL_NAMES)
    print(f"本轮可用工具：{[t['工具名'] for t in param_tools]}")
    openai_tools = _to_openai_tools(param_tools)

    prompts = [
        "帮我创建用户：姓名张三，邮箱 zhangsan@example.com，年龄 28，标签是「学生」和「篮球」，设置为已激活。",
        "25 摄氏度等于多少华氏度？",
    ]
    for prompt in prompts:
        print()
        print(f"[用户] {prompt}")
        # 每个问题都用一份干净的、只含 system + user 的历史
        run_agent_loop([{"role": "user", "content": prompt}], openai_tools, system_prompt)

### 2.5 跑起来

三个演示依次执行。回看每次输出末尾那行 `[主循环共转 N 圈…]`：

- 转 2 圈 = 模型调一次工具、拿到结果后再要一次（这就是 Function Call 的往返）；
- 演示 1 里模型可能反复调同一个工具、一直撞到 10 圈上限 —— 那正是「循环上限」存在的意义。

In [ ]:
if READY:
    print(f"模型：{settings.model_name}    接口：{settings.base_url}")
    print(f"模块顶部 tools 共 {len(tools)} 个（= list_tools() 扫到的全部工具）")
    print()

    demo_math_few_shot()        # 课案《实例》原样（带 Few-shot）
    demo_math_plain()           # 同一道题去掉 Few-shot，复现课案预期轨迹
    demo_param_types()          # 课案《参数类型》9 种参数的实战

    print()
    print("=" * 62)
    print("三个演示跑完。回看每次输出末尾那行 [主循环共转 N 圈…]：")
    print("  - 转 2 圈 = 模型调一次工具、拿到结果后再要一次（这就是 Function Call 的往返）；")
    print("  - 演示 1 里模型反复调同一个工具、一直撞到 10 圈上限，正是「循环上限」存在的意义。")
    print("=" * 62)
else:
    print("[跳过] 前置条件不足，路径 A 的三个演示未执行")

### 预期输出

```text
模型：deepseek-flash    接口：https://api.deepseek.com
模块顶部 tools 共 13 个（= list_tools() 扫到的全部工具）

==============================================================
演示 1：课案《实例》原样（Few-shot 历史 + 用户问 2+4*6）
==============================================================
本轮可用工具：['add_tool', 'div_tool', 'mul_tool', 'sub_tool']

调用工具: mul_tool, 参数: {'a': 4, 'b': 6}, 结果: 24
调用工具: add_tool, 参数: {'a': 2, 'b': 24}, 结果: 26
结束
最终答案: 2+24=26，所以2+4*6=26。
[主循环共转 3 圈，执行 2 次工具调用]

==============================================================
演示 2：去掉 Few-shot，首轮 tool_choice='required'（稳定复现课案轨迹）
==============================================================
  [降级] 本端点不支持 tool_choice='required'（BadRequestError: Error code: 400 - {'error': {'message': 'Thinking mode does not support this tool_choice',）
         改用 tool_choice='auto' 重试：本轮是否调工具由模型自行决定。
调用工具: mul_tool, 参数: {'a': 4, 'b': 6}, 结果: 24
调用工具: add_tool, 参数: {'a': 2, 'b': 24}, 结果: 26
结束
最终答案: 计算过程：先算乘法 **4 × 6 = 24**，再算加法 **2 + 24 = 26**。

所以 **2 + 4 × 6 = 26**。
[主循环共转 3 圈，执行 2 次工具调用]

==============================================================
演示 3：参数类型（string / integer / boolean / array / enum 真的被填对了）
==============================================================
本轮可用工具：['calc_price_tool', 'check_stock_tool', 'convert_temperature_tool', 'create_user_tool', 'describe_user_tool', 'format_username_tool', 'set_nickname_tool', 'summarize_tags_tool', 'toggle_feature_tool']

[用户] 帮我创建用户：姓名张三，邮箱 zhangsan@example.com，年龄 28，标签是「学生」和「篮球」，设置为已激活。
调用工具: create_user_tool, 参数: {'name': '张三', 'email': 'zhangsan@example.com', 'age': 28, 'is_active': True, 'tags': ['学生', '篮球']}, 结果: {"已创建用户": {"name": "张三", "email": "zhangsan@example.com", "age": 28, "is_active": true, "tags": ["学生", "篮球"]}}
结束
最终答案: 用户已成功创建 ✅

| 字段 | 值 |
|------|-----|
| 姓名 | 张三 |
| 邮箱 | zhangsan@example.com |
| 年龄 | 28 |
| 标签 | 学生、篮球 |
| 状态 | 已激活 |
……
[主循环共转 2 圈，执行 1 次工具调用]

[用户] 25 摄氏度等于多少华氏度？
调用工具: convert_temperature_tool, 参数: {'value': 25, 'unit': 'celsius'}, 结果: 25°C = 77.0°F
结束
最终答案: 25 摄氏度等于 **77 华氏度**（25°C = 77.0°F）。
[主循环共转 2 圈，执行 1 次工具调用]

==============================================================
三个演示跑完。回看每次输出末尾那行 [主循环共转 N 圈…]：
  - 转 2 圈 = 模型调一次工具、拿到结果后再要一次（这就是 Function Call 的往返）；
  - 演示 1 里模型反复调同一个工具、一直撞到 10 圈上限，正是「循环上限」存在的意义。
==============================================================
```

**三个必须逐字读的现象**：

1. **本机确确实实走了降级**：演示 2 那两行 `[降级] … 400 … Thinking mode does not support
   this tool_choice` 是同一个 notebook 在支持强制工具选择的端点上**不会出现**的输出；
   它的作用是「**说出来，别静默吞掉**」—— 语义确实从「逼模型调工具」变成了「模型自己决定」。
2. **演示 1 这次没撞到 10 圈上限**（转了 3 圈就收工）。源码注释里「演示 1 会一直撞上限」
   是一种**可能会出现**的情况，不是每次都会 —— 工具描述与 Few-shot 的共同作用依模型而异，
   这也是这一格要保留 `[主循环共转 N 圈]` 这行计数的原因：**让实际行为自己说话**。
3. **演示 3 把参数类型全填对了**：`age` 是 int `28`（不是 `28.0`）、
   `is_active` 是 `True`、`tags` 是 `['学生', '篮球']`、`unit` 落在 `enum` 里的 `'celsius'` ——
   这正是 `TOOL_PARAMETERS` 里类型写准了的结果。

> ⚠️ 这整块输出**由模型决定，是本节最不稳定的一段**：循环转几圈、调几次工具、
> 每一句「最终答案」怎么措辞，**每次运行都会变**（连演示 1 是不是撞到 10 圈上限都不固定）；
> 上面只是我这次跑出来的**实测值**，**别逐字比对**。
> 不会变的是骨架：演示标题 → `调用工具: … 参数: … 结果: …` → `结束` → `最终答案: …`
> → `[主循环共转 N 圈，执行 M 次工具调用]`。**要核对的是这个骨架，以及参数类型填得对不对。**

## 3. 完整版 · 路径 B：LangChain 的两条路线

课案给出了**两条路线**，本节都实现并各跑一遍：

| 路线 | 写法 | 循环在谁手里 | 适合 |
|---|---|---|---|
| 路线一 `create_agent` | `create_agent(model, tools, system_prompt)` | **框架**（LangGraph 图） | 官方推荐、开箱即用 |
| 路线二 `bind_tools` | `llm.bind_tools(tools)` + 自己写 `for` | **你自己** | 想在框架里保留手动控制 |

一句话：**`create_agent` ≈ `bind_tools` + 一个写好的主循环**。

### 3.1 `@tool` 装饰器定义工具

与路径 A 的裸函数相比只多了两样：`@tool` 装饰器（自动导出 name / description / args_schema），
以及**类型注解** `a: float, b: float -> float`（参数类型的推断依据，
没有注解 LangChain 生成不出 `parameters`，模型就不知道该填什么类型）。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool


@tool
def add_tool(a: float, b: float) -> float:
    """
    返回a+b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a+b的结果
    """
    return a + b


@tool
def sub_tool(a: float, b: float) -> float:
    """
    返回a-b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a-b的结果
    """
    return a - b


@tool
def mul_tool(a: float, b: float) -> float:
    """
    返回a*b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a*b的结果
    """
    return a * b


@tool
def div_tool(a: float, b: float) -> float:
    """
    返回a/b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a/b的结果
    """
    return a / b


# 课案原文：tools_list = [tools.add_tool, tools.sub_tool, tools.mul_tool, tools.div_tool]
tools_list = [add_tool, sub_tool, mul_tool, div_tool]

# 工具名 → 工具对象：bind_tools 路线里，模型只回传工具名，程序得自己找回工具
TOOL_MAP = {t.name: t for t in tools_list}

print(f"@tool 版本的工具：{[t.name for t in tools_list]}")
print(f"mul_tool 的 Schema（LangChain 自动生成的）：{mul_tool.args}")

### 预期输出

```text
@tool 版本的工具：['add_tool', 'sub_tool', 'mul_tool', 'div_tool']
mul_tool 的 Schema（LangChain 自动生成的）：{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
```

上面那行 `args` 就是「**类型注解变成了 JSON Schema**」的直观证据 ——
`a: float, b: float` 被翻译成了 `{"type": "number"}`，
与路径 A 里手写的 `SCHEMA_MATH_A` / `SCHEMA_MATH_B` 是同一件事，只是不用你写了。

> ⚠️ 上面这些 `@tool` 与路径 A 里**同名**（`add_tool` / `sub_tool` / `mul_tool` / `div_tool`）。
> 这里的重定义是**有意**的：路径 A 用「裸函数 + 手写 Schema」，
> 路径 B 用「`@tool` 对象 + 自动 Schema」，同名正好体现「工具本体没变、包装换了」。
> 前面 `TOOL_REGISTRY` / `tools_jxsd` 里保存的是**原来的函数对象**，不受这次重定义影响。

### 3.2 路线一：`create_agent`

`create_agent` 返回的是一个 **LangGraph 图**：`model` 节点 + `tools` 节点 + 条件边。
模型有 `tool_calls` 就走向 `tools` 节点执行，执行完再回到 `model` 节点，
直到模型不再要工具为止 —— 这正是路径 A 里手写的那个循环。

`print_trace` 遍历 `result["messages"]`，把这条 `ai → tool → ai` 的链摊开给你看。
课案原文是 `print(f"Content: {msg}")`，直接把整条消息对象打出来 ——
里面塞满了 `response_metadata` / `token_usage` / `id`，一屏根本看不完；
这里精简成「角色 + 内容 + 工具调用」三样关键信息（课案想看的东西一样不少）。

In [ ]:
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

agent = create_agent(
    model=llm,
    tools=tools_list,
    system_prompt="你是一个助手，可以帮助用户进行数学计算。",
)


def build_few_shot_messages() -> list[dict]:
    """
    课案的 Few-shot 历史对话（与路径 A 里那段完全一致）

    注意这里用的是**标准 OpenAI 消息格式**（assistant 带 tool_calls、
    tool 带 tool_call_id），LangChain 能直接吃下去——它会自动转成
    AIMessage / ToolMessage。这也是 LangChain 好用的地方：
    消息格式与 OpenAI 协议基本兼容，不用手写适配层。
    """
    return [
        {"role": "user", "content": "8*2-9"},
        {"role": "assistant", "content": "要计算8*2-9。首先计算8*2。", "tool_calls": [
            {"id": "call_1", "type": "function",
             "function": {"name": "mul_tool", "arguments": json.dumps({"a": 8, "b": 2})}}
        ]},
        {"role": "tool", "tool_call_id": "call_1", "content": "16"},
        {"role": "assistant", "content": "8*2=16，现在计算16-9。", "tool_calls": [
            {"id": "call_2", "type": "function",
             "function": {"name": "sub_tool", "arguments": json.dumps({"a": 16, "b": 9})}}
        ]},
        {"role": "tool", "tool_call_id": "call_2", "content": "7"},
        {"role": "assistant", "content": "16-9=7，所以8*2-9=7"},
    ]


def print_trace(result: dict) -> None:
    """
    课案的打印方式：遍历 result["messages"]，逐条打印角色和内容

    「最终答案」其实只是最后一条消息；中间那些 ai → tool → ai 才是
    create_agent 自动帮我们跑完的工具循环，翻一遍就能看清它做了什么。
    """
    print("最终答案（完整消息链）：")
    for msg in result["messages"]:
        # msg.type 取值：human / ai / tool / system
        #   ai 带 tool_calls    → 模型在要工具
        #   tool               → 框架自动执行工具后的结果回填
        #   最后一条 ai 无 tool_calls → 才是真正的自然语言答案
        print(f"Role: {msg.type}")
        if getattr(msg, "tool_calls", None):
            for call in msg.tool_calls:
                print(f"   → 提出调用 {call['name']}({call['args']})  id={call['id']}")
        if msg.content:
            print(f"Content: {msg.content}")
        print("-" * 50)

### 3.3 路线二：`bind_tools`（手动循环的 LangChain 写法）

这一格其实是**路径 A 主循环的 LangChain 版**，逐行对照只差三处：

1. `tools` 不用手写 JSON Schema —— 直接传 `@tool` 对象，LangChain 负责转换；
2. `ai.tool_calls` 里的 `args` **已经是 dict**，不用 `json.loads`（原生 SDK 给的是字符串）；
3. 回填结果用 `ToolMessage` 对象，不用手拼 `{"role": "tool", ...}` 字典。

循环结构、循环上限、退出条件，三者一模一样。

In [ ]:
def run_with_bind_tools(user_input: str, max_turns: int = 10) -> str | None:
    """
    把工具绑到模型上，然后自己写循环

    与 agent_openai_jxsd.py 的主循环逐行对照，差别只有三处：
        1. tools 不用手写 JSON Schema —— 直接传 @tool 对象，LangChain 负责转换；
        2. ai.tool_calls 里的 args **已经是 dict**，不用 json.loads（原生 SDK 给的是字符串）；
        3. 回填结果用 ToolMessage 对象，不用手拼 {"role": "tool", ...} 字典。
    循环结构、循环上限、退出条件，三者一模一样。
    """
    bound_llm = llm.bind_tools(tools_list)

    messages = [
        SystemMessage("你是一个助手，可以帮助用户进行数学计算。"),
        HumanMessage(user_input),
    ]

    for _ in range(max_turns):
        ai = bound_llm.invoke(messages)
        messages.append(ai)

        # 模型没提工具调用 → 这就是最终答案
        if not ai.tool_calls:
            print(f"最终答案: {ai.content}")
            return ai.content

        for call in ai.tool_calls:
            # LangChain 已经把工具调用规整成 {"name": ..., "args": {...}, "id": ...}
            # args 直接就是 dict，这是它比原生 SDK 省事的地方
            result = TOOL_MAP[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
            print(f"调用工具: {call['name']}, 参数: {call['args']}, 结果: {result}")
    else:
        print(f"⚠️ 已循环 {max_turns} 次仍未得到最终答案，强制结束（防御上限生效）")
    return None

### 3.4 两条路线各跑一遍

路线一跑两次：一次带 Few-shot（课案原样），一次干净版。
与路径 A 演示 1/2 的原因相同：**当前模型对这段 Few-shot 并不买账**，
去掉示例后工具循环反而跑得干净利落 —— 这是同一现象在两个框架上的复现。

In [ ]:
if READY:
    print("=" * 62)
    print("路线一：create_agent（课案原样，带 Few-shot 历史）")
    print("=" * 62)
    messages = build_few_shot_messages()
    messages.append({"role": "user", "content": "2+4*6"})
    result = agent.invoke({"messages": messages})
    print_trace(result)

    print()
    print("=" * 62)
    print("路线一（干净版）：去掉 Few-shot，同一道题")
    print("=" * 62)
    result = agent.invoke({"messages": [{"role": "user", "content": "2+4*6"}]})
    print_trace(result)

    print()
    print("=" * 62)
    print("路线二：bind_tools（自己写循环，与原生 SDK 逐行对照）")
    print("=" * 62)
    run_with_bind_tools("2+4*6")

    print()
    print("=" * 62)
    print("两条路线对照（课案《与 OpenAI API 实现的对比》表以注释形式保留在源码里）：")
    print("=" * 62)
    print("  create_agent —— 一行创建，工具循环全自动（结果看 result['messages'] 里的 ai→tool→ai）")
    print("  bind_tools   —— 自己写循环；args 已经是 dict，比原生 SDK 少一步 json.loads")
    print("  共同点       —— JSON Schema 不用手写：@tool 把 docstring + 类型注解自动转成 parameters")
    print("  共同点       —— 初始化从 openai.Client() 变成 ChatOpenAI / init_chat_model")
else:
    print("[跳过] 前置条件不足，路径 B 的两条路线演示未执行")

### 预期输出

```text
==============================================================
路线一：create_agent（课案原样，带 Few-shot 历史）
==============================================================
最终答案（完整消息链）：
Role: human
Content: 8*2-9
--------------------------------------------------
Role: ai
   → 提出调用 mul_tool({'a': 8, 'b': 2})  id=call_1
Content: 要计算8*2-9。首先计算8*2。
--------------------------------------------------
Role: tool
Content: 16
--------------------------------------------------
Role: ai
   → 提出调用 sub_tool({'a': 16, 'b': 9})  id=call_2
Content: 8*2=16，现在计算16-9。
--------------------------------------------------
Role: tool
Content: 7
--------------------------------------------------
Role: ai
Content: 16-9=7，所以8*2-9=7
--------------------------------------------------
Role: human
Content: 2+4*6
--------------------------------------------------
Role: ai
   → 提出调用 mul_tool({'a': 4, 'b': 6})  id=call_00_sd1wUEh1Wtvlpmce3YaD8137
Content: 要计算2+4*6。根据运算顺序，先计算4*6。
--------------------------------------------------
Role: tool
Content: 24.0
--------------------------------------------------
Role: ai
   → 提出调用 add_tool({'a': 2, 'b': 24})  id=call_00_GSwcu8D8770PC09anjWp5162
Content: 4*6=24，现在计算2+24。
--------------------------------------------------
Role: tool
Content: 26.0
--------------------------------------------------
Role: ai
Content: 2+4*6=26
--------------------------------------------------

==============================================================
路线一（干净版）：去掉 Few-shot，同一道题
==============================================================
最终答案（完整消息链）：
Role: human
Content: 2+4*6
--------------------------------------------------
Role: ai
   → 提出调用 mul_tool({'a': 4, 'b': 6})  id=call_00_9U9M9DLdjl1iVnKhfH9b7359
--------------------------------------------------
Role: tool
Content: 24.0
--------------------------------------------------
Role: ai
   → 提出调用 add_tool({'a': 2, 'b': 24})  id=call_00_j9UZIG85U2YzhptjgZfy3571
--------------------------------------------------
Role: tool
Content: 26.0
--------------------------------------------------
Role: ai
Content: 计算过程（先乘除后加减）：

- 先算乘法：4 × 6 = 24
- 再算加法：2 + 24 = **26**

所以，2 + 4 × 6 = **26**。
--------------------------------------------------

==============================================================
路线二：bind_tools（自己写循环，与原生 SDK 逐行对照）
==============================================================
调用工具: mul_tool, 参数: {'a': 4, 'b': 6}, 结果: 24.0
调用工具: add_tool, 参数: {'a': 2, 'b': 24}, 结果: 26.0
最终答案: 根据运算顺序，先算乘法再算加法：

- \(4 \times 6 = 24\)
- \(2 + 24 = 26\)

所以结果是 **26**。

==============================================================
两条路线对照（课案《与 OpenAI API 实现的对比》表以注释形式保留在源码里）：
==============================================================
  create_agent —— 一行创建，工具循环全自动（结果看 result['messages'] 里的 ai→tool→ai）
  bind_tools   —— 自己写循环；args 已经是 dict，比原生 SDK 少一步 json.loads
  共同点       —— JSON Schema 不用手写：@tool 把 docstring + 类型注解自动转成 parameters
  共同点       —— 初始化从 openai.Client() 变成 ChatOpenAI / init_chat_model
```

**三处和路径 A 逐行对照出来的差别**：

1. **`ai → tool → ai` 是框架自己转的**：路线一那次 `agent.invoke` 内部就完成了
   「要工具 → 执行 → 回填 → 再问」；路径 A 里这些是我们手写的。
2. **`args` 已经是 dict**：`mul_tool({'a': 4, 'b': 6})` —— 没有 `json.loads` 这一步；
   而路径 A 拿到的是 `'{"a": 4, "b": 6}'` 这样的字符串。
3. **工具结果变成了 `24.0` / `26.0`**：因为 `@tool` 上的类型注解是 `float`，
   `str(24.0)` 就是 `"24.0"` —— 数值等价，只是写法与路径 A 的 `24` / `26` 不同。

> Few-shot 那一段在路线一里**完整复现了课案示例**（`Role: human/ai/tool` 那 6 条），
> 说明 LangChain 能直接吃标准 OpenAI 消息格式，不用手写适配层。

> ⚠️ 上面整块输出的**正文与工具调用顺序都由模型决定**，**每次运行都不同**：
> `call_00_…` 那串 id 是随机 UUID，最后那条 `Content:` 的措辞每次也不同 ——
> 这只是我这次跑出来的**实测值**，**别逐字比对**。
> 稳定不变的是骨架：`Role: human → ai(带 tool_calls) → tool → ai(最终答案)` 这个 `ai → tool → ai` 链，
> 以及路线二那三行 `调用工具: … / 最终答案: …`。

## 4. 完整版 · 路径 C：DeepAgents 的内置能力

要看清这一节，关键是分清「一样」和「不一样」：

**一样的地方**（课案重点强调的）

- 工具定义：还是 `@tool` 装饰器，一个字都不用改；
- 调用方式：还是 `agent.invoke({"messages": [...]})`，连返回结构都一样；
- 模型对象：还是同一个 `init_chat_model`。

**不一样的地方**（课案对比表里的「额外能力」）

| 中间件 | 白送的工具 | 作用 |
|---|---|---|
| `FilesystemMiddleware` | `ls` / `read_file` / `write_file` / `edit_file` | 模型可以直接读写文件 |
| `TodoListMiddleware` | `write_todos` | 把复杂任务拆成待办清单逐步勾掉 |
| 子 Agent 委派 | `task` | 把子任务外包给临时的子 Agent |
| 上下文压缩 | —（自动） | 对话太长时自动摘要，避免撑爆上下文窗口 |

一句话（课案原文）：**`create_deep_agent` = `create_agent` + 全套预装中间件**。

### 4.1 工具与模型：同一批工具，写法不变

In [ ]:
from deepagents import create_deep_agent


@tool
def add_tool(a: float, b: float) -> float:
    """返回 a + b 的结果"""
    return a + b


@tool
def sub_tool(a: float, b: float) -> float:
    """返回 a - b 的结果"""
    return a - b


@tool
def mul_tool(a: float, b: float) -> float:
    """返回 a * b 的结果"""
    return a * b


@tool
def div_tool(a: float, b: float) -> float:
    """返回 a / b 的结果"""
    return a / b


# ---------- 模型 ----------
model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# 同一批工具，三个版本共用（OpenAI 原生 / LangChain / DeepAgents）
tools = [add_tool, sub_tool, mul_tool, div_tool]

### 4.2 唯一区别：`create_agent` → `create_deep_agent`

课案原文的调用与 LangChain 那节**逐字相同**，只换了函数名。

`print_messages` 保持课案的结构，做了两点可读性处理（实测必需）：

1. 把「模型提出工具调用」那一步展开成一行，否则 `tool_calls`
   混在整条消息对象里根本看不出来；
2. 折叠**完全重复**的消息：本机模型有时会把同一批调用反复提好几遍，
   原样打印会刷屏几十行，看不出主线索。

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=tools,
    system_prompt="你是一个助手，可以帮助用户进行数学计算。",
)


def print_messages(result: dict) -> None:
    """
    课案原文的打印方式：
        for msg in result["messages"]:
            print(f"{msg.type}: {msg}")

    这里保持课案的结构，做了两点可读性处理（实测必需）：
        1. 把「模型提出工具调用」那一步展开成一行，否则 tool_calls
           混在整条消息对象里根本看不出来；
        2. 折叠**完全重复**的消息：本机模型有时会把同一批调用反复提好几遍，
           原样打印会刷屏几十行，看不出主线索。
    """
    printed = set()          # 去重指纹：重复出现的同一动作只打第一次

    for msg in result["messages"]:
        calls = getattr(msg, "tool_calls", None) or []
        # 给「带工具调用的 ai 消息」和「工具结果消息」各算一个指纹
        if calls:
            fingerprint = ("call", tuple((c["name"], str(c["args"])) for c in calls))
        elif getattr(msg, "name", None):
            fingerprint = ("tool", msg.name, str(msg.content))
        else:
            fingerprint = None
        if fingerprint is not None:
            if fingerprint in printed:
                continue     # 重复动作，跳过（第一次已经打过了）
            printed.add(fingerprint)

        print(f"{msg.type}: {msg.content if msg.content else ''}")
        # 关键：DeepAgents 内置的工具（write_todos / write_file ...）也会以
        # ToolMessage 的形式出现在这条消息链里——它们不是你定义的，
        # 是中间件自动加进来的，这正是「额外能力」的直观证据
        for call in calls:
            print(f"    → 提出调用 {call['name']}({call['args']})")
        if getattr(msg, "name", None):
            # ToolMessage.name = 是哪个工具的执行结果
            print(f"    ← 工具 {msg.name} 的执行结果")
        print("-" * 50)

### 4.3 两个演示

- **演示 1** 课案原样：只问算数，看 API 是否真的完全一致；
- **演示 2** 用上「额外能力」：先打印中间件装进来的内置工具清单
  （`write_file` / `write_todos` 这些**我们一个都没定义**），
  再让模型真的调一次 `write_file` —— 这跟 `create_agent` 是本质差别，
  后者遇到这种要求只能干瞪眼，因为它手里根本没有文件工具。

> `write_file` 写进的是 DeepAgents 的**状态后端**（默认 `StateBackend`，存在图状态里），
> **不会真的在磁盘上建文件**，所以放心跑，不会污染仓库。
> 想落到真实磁盘，得换成 `FilesystemBackend` / 自定义 backend。

In [ ]:
if READY:
    print("=" * 62)
    print("演示 1：课案原样 —— 只问算数，看 API 是否真的完全一致")
    print("=" * 62)
    result = agent.invoke({"messages": [{"role": "user", "content": "2+4*6"}]})
    print_messages(result)
    # 预期轨迹与前面几节相同：mul_tool(4,6)=24 → add_tool(2,24)=26 → 最终答案 26
    # 唯一多出来的，是 DeepAgents 自己那套中间件在背后运转（本轮用不上就不会出现）
    # ⚠️ 本机实测：当前模型有时只调一次 mul_tool 就收工，甚至给出「结果是 24」这种
    #    漏掉加法的错误答案。要分清责任——协议和循环都是对的（工具确实被调用了、
    #    结果也确实回填了），错在模型自己的推理。

    print()
    print("=" * 62)
    print("演示 2：用上课案对比表里说的「额外能力」（任务拆解 + 文件系统）")
    print("=" * 62)
    # 先看看 create_deep_agent 到底白送了哪些工具（这是课案对比表里
    # 「额外能力：内置文件系统、任务拆解」的可核对证据）：
    # 下面这些工具我们一个都没定义，全是中间件装进来的
    print("create_deep_agent 自动装上的内置工具：")
    try:
        from deepagents.middleware import FilesystemMiddleware
        from langchain.agents.middleware import TodoListMiddleware
        # FilesystemMiddleware 负责文件系统那一套
        print(f"  文件系统（FilesystemMiddleware）：{[t.name for t in FilesystemMiddleware().tools]}")
        # TodoListMiddleware 负责任务拆解：write_todos
        print(f"  任务拆解（TodoListMiddleware）  ：{[t.name for t in TodoListMiddleware().tools]}")
    except Exception as exc:                      # noqa: BLE001 —— 中间件属于框架内部实现，版本变动时不该炸掉整个演示
        print(f"  （当前 deepagents 版本取不到中间件清单：{exc}）")
    print()

    # 注意这条 prompt：里面提到的 write_file 我们**一个都没定义**——
    # 它是 create_deep_agent 预装中间件自带的工具。
    #
    # 实测经验（多次跑下来结论很一致）：
    #   - 把要用的工具名点出来 → 模型基本都会照做，能稳定看到中间件工具被执行；
    #   - 一句话里塞三件事（列清单 + 算数 + 写文件）→ 模型经常只在正文里
    #     「复述」它打算调用哪个工具，却不真的发 tool_calls，甚至直接收工。
    #   所以这里刻意拆成一步一个小任务，保证演示能跑出结果。
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
                       "请调用 write_file 工具，把 '2+4*6=26' 写入 result.md"}]},
        # 中间件会多跑好几轮，默认递归上限容易撞到，这里放宽到 50
        # （与本目录既有 deepagents 示例一致）
        config={"recursion_limit": 50},
    )
    print_messages(result)

    print()
    print("=" * 62)
    print("课案《LangChain create_agent vs DeepAgents create_deep_agent》对比表")
    print("（原文以注释形式保留在本文件末尾）")
    print("=" * 62)
    print("  create_agent      —— 工具定义 @tool / 调用 agent.invoke() / 无额外能力 / 精准控制")
    print("  create_deep_agent —— 工具定义完全相同 / 调用方式完全相同")
    print("                       额外：内置文件系统、任务拆解、子Agent委派、上下文压缩")
    print("                       适用：快速搭建，想立刻干活")
    print()
    print("三个版本对照小结：")
    print("  agent_openai_jxsd.py     —— 手写循环，理解原理")
    print("  agent_langchain_jxsd.py  —— create_agent / bind_tools，工具循环自动化")
    print("  agent_deepagents_jxsd.py —— create_deep_agent，在工具循环之上再送一套中间件")
else:
    print("[跳过] 前置条件不足，路径 C 的两个演示未执行")

### 预期输出

```text
==============================================================
演示 1：课案原样 —— 只问算数，看 API 是否真的完全一致
==============================================================
human: 2+4*6
--------------------------------------------------
ai: 
    → 提出调用 mul_tool({'a': 4, 'b': 6})
--------------------------------------------------
tool: 24.0
    ← 工具 mul_tool 的执行结果
--------------------------------------------------
ai: 
    → 提出调用 add_tool({'a': 2, 'b': 24})
--------------------------------------------------
tool: 26.0
    ← 工具 add_tool 的执行结果
--------------------------------------------------
ai: 2 + 4 × 6 = 2 + 24 = **26**
--------------------------------------------------

==============================================================
演示 2：用上课案对比表里说的「额外能力」（任务拆解 + 文件系统）
==============================================================
create_deep_agent 自动装上的内置工具：
  文件系统（FilesystemMiddleware）：['ls', 'read_file', 'write_file', 'edit_file', 'delete', 'glob', 'grep', 'execute']
  任务拆解（TodoListMiddleware）  ：['write_todos']

human: 请调用 write_file 工具，把 '2+4*6=26' 写入 result.md
--------------------------------------------------
ai: 
    → 提出调用 write_file({'file_path': 'result.md', 'content': '2+4*6=26\n'})
--------------------------------------------------
tool: Updated file /result.md
    ← 工具 write_file 的执行结果
--------------------------------------------------
ai: 已完成：已将 `2+4*6=26` 写入 `result.md`。

（验算：先乘后加，4×6=24，2+24=26，等式成立。）
--------------------------------------------------

==============================================================
课案《LangChain create_agent vs DeepAgents create_deep_agent》对比表
（原文以注释形式保留在本文件末尾）
==============================================================
  create_agent      —— 工具定义 @tool / 调用 agent.invoke() / 无额外能力 / 精准控制
  create_deep_agent —— 工具定义完全相同 / 调用方式完全相同
                       额外：内置文件系统、任务拆解、子Agent委派、上下文压缩
                       适用：快速搭建，想立刻干活

三个版本对照小结：
  agent_openai_jxsd.py     —— 手写循环，理解原理
  agent_langchain_jxsd.py  —— create_agent / bind_tools，工具循环自动化
  agent_deepagents_jxsd.py —— create_deep_agent，在工具循环之上再送一套中间件
```

**把「额外能力」钉死成证据的三行**：

1. **`FilesystemMiddleware` 那 8 个工具**（`ls` / `read_file` / `write_file` / `edit_file` /
   `delete` / `glob` / `grep` / `execute`）—— **我们一个都没定义**，
   它们是被中间件装进来的。注意：不同 deepagents 版本这一串名字会变，
   所以源码里写了 `try/except` 兜底（取不到就打印一行说明，而不是整格炸掉）。
2. **`write_file` 真的被执行了**：工具消息 `Updated file /result.md` 就是证据；
   但 `result.md` **没有出现在仓库里** —— 它写在 DeepAgents 的状态后端里。
3. **`ai:` 后面是空的**：那一轮 assistant 只发了工具调用、没有正文，
   正是路径 A 里 `if not message.tool_calls` 那个判断所对应的「中间轮」。

> ⚠️ 上面整块输出**由模型决定，每次运行都不同**：模型可能只调一次工具就收工、
> 可能给出漏掉加法的错答案、也可能改用 `write_todos` 而不写文件，
> 最后那条 `ai:` 的措辞更是每次都不一样 —— 这只是我这次跑出来的**实测值**，**别逐字比对**。
> 另外中间件那两串工具名**会随 deepagents 版本变化**（源码里 `try/except` 就是为它留的）。
> 稳定不变的是骨架：`human → ai(带 tool_calls) → tool → … → ai(最终答案)`。

## 5. 三路径横向对比（本节的核心结论）

把这一节看过的三份代码并排放，结论就是下面这张表：

| 维度 | A 原生 OpenAI SDK | B LangChain `create_agent` | C DeepAgents `create_deep_agent` |
|---|---|---|---|
| **代码量** | 主循环约 80 行 + 手写 JSON Schema | 创建 Agent **1 行** + `@tool` | 创建 Agent **1 行** + `@tool`（与 B 完全相同） |
| **谁管循环** | **你**（`for turn in range(max_turns)`） | 框架（LangGraph 的 model ↔ tools 图） | 框架（同 B，再叠中间件） |
| **谁管工具** | **你**：`json.loads(arguments)` → `TOOL_REGISTRY[name](**args)` → 手拼 `{"role": "tool", ...}` | 框架：`@tool` 生成 Schema，自动执行 + 自动回填 `ToolMessage` | 框架：**与 B 完全相同** |
| **工具结果形态** | `arguments` 是 **JSON 字符串**，要自己解析 | `call["args"]` 已经是 **dict** | 同 B |
| **看得见中间过程吗** | 完全可见（每轮自己打印） | 过程被封装，只能回头翻 `result["messages"]` | 同 B（消息链更长，中间件会插自己的步骤） |
| **额外能力** | 无 | 无（但可以自己加 middleware） | 文件系统 / 任务拆解 / 子 Agent 委派 / 上下文压缩 |
| **适用场景** | 理解原理、需要完全掌控、接非 OpenAI 协议 | 大多数业务：轻量、够用、想精准控制 | 复杂长程多步任务：想立刻干活 |
| **本节的坑** | 必须自己设循环上限、`tool_call_id` 必须对应 | Few-shot 效果依模型而异 | 同 B；`recursion_limit` 要放宽 |

三条路径的关系可以写成两个等式：

```text
create_agent      ≈ bind_tools + 一个写好的主循环
create_deep_agent =  create_agent + 全套预装中间件
```

**怎么选**：

- 想搞懂 Agent 到底怎么跑 → 把路径 A 那段循环亲手写一遍，比读十篇文章有用；
- 日常业务 → 路径 B，一行创建 + `@tool` 装饰器，性价比最高；
- 任务本身就长（多步、要读写文件、要拆清单、要派子任务）→ 路径 C，别自己造中间件。

## 小结

1. **三条路径做的是同一件事**：把 `tools` 描述发给模型 → 模型返回 `tool_calls`
   → 执行本地函数 → 把结果回填历史 → 再问一轮，直到模型不再要工具。
   差别只在**这段循环由谁写、过程由谁管**。
2. **`@tool` 装饰器把「两份东西」合成了一份**：路径 A 要同时维护
   「Python 函数」和「手写 JSON Schema」，两边靠函数名对齐；
   路径 B/C 只需写函数，`docstring` + 类型注解自动变成 `description` + `parameters`。
3. **工具定义在三个版本里一个字都不用改** —— 这是课案反复强调的一句话，
   本节的路径 B/C 两段 `@tool` 代码就是它的证据。
4. **降级要说出来**：本机 `deepseek-flash` 不支持 `tool_choice="required"`，
   演示 2 走的是 `[降级]` 分支。这不是代码错，而是**端点能力差异**；
   代码把这件事打印出来而不是静默吞掉，才是可交付的做法。
5. **循环上限不是摆设**：路径 A 的 `range(10)` 与路径 C 的 `recursion_limit: 50`
   都是同一件事 —— 模型完全可能陷在「反复调同一个工具」的怪圈里。

## 常见坑

| 症状 | 原因 | 解法 |
|---|---|---|
| `400 Thinking mode does not support this tool_choice` | 思考模型不支持强制工具选择 | 退回 `tool_choice="auto"`（本节演示 2 的降级分支就是这个） |
| `400` 说 schema 不合法 | `strict: True` 要求每个 object 都带 `additionalProperties: False`，且 `required` 覆盖全部字段 | 少哪条补哪条；或把 `strict` 改成 `False` |
| 模型只在正文里**复述**「我要调用 xxx」却不动手 | 模型随性（`auto` 下常见） | 首轮用 `required`（端点支持时）；或把任务拆成一步一个小任务 |
| `unexpected keyword argument` | JSON Schema 里的参数名与 Python 形参名不一致 | 两边名字严格对齐（这是模型与程序唯一的对齐键） |
| `KeyError` / `AttributeError` 炸掉主循环 | 模型漏字段或报了个不存在的工具名 | 工具里用 `.get` 兜底；分发用白名单注册表 + `try/except` 回传错误文本 |
| 同一个工具被反复调用，直到撞到上限 | 模型陷入循环 | 保留循环上限（`range(10)` / `recursion_limit`），别为了「跑通」把它去掉 |
| `ModuleNotFoundError: tools` | 照抄归档脚本，但归档区不在 `sys.path` 上 | 本节的做法：把工具定义内联，再装配成同名模块注册进 `sys.modules` |
| 工具列表里凭空多出一个「不存在的工具」 | `list_tools()` 靠 `endswith("_tool")` 扫模块，辅助函数 `call_tool` 也被扫进去了（本节实测第 14 个工具就是它） | 装配模块时**显式列出真正的工具函数**，别用「所有 `*_tool` 名字」 |
| `recursion_limit` 报错 / 图跑到一半停 | DeepAgents 中间件会多跑好几轮 | `config={"recursion_limit": 50}` |

## 官方链接

- LangChain Agents（`create_agent` / middleware）：<https://docs.langchain.com/oss/python/langchain/agents>
- LangChain 工具（`@tool` 装饰器）：<https://docs.langchain.com/oss/python/langchain/tools>
- DeepAgents 总览（`create_deep_agent` 与内置中间件）：<https://docs.langchain.com/oss/python/deepagents/overview>
- DeepAgents 快速上手：<https://docs.langchain.com/oss/python/deepagents/quickstart>
- OpenAI Function Calling 指南（`tools` / `tool_choice` / `strict`）：<https://developers.openai.com/api/docs/guides/function-calling>
- OpenAI Chat Completions API 参考：<https://platform.openai.com/docs/api-reference/chat>

> 归档源码里还留着一张《Chat Completions API vs Responses API》的对比表，
> 其中一行正是本节的起点：「工具执行循环需客户端自行编写」。
> 换成 Responses API，平台会替你编排多步工具调用 —— 方向上与
> `create_agent` / `create_deep_agent` 越包越省事是同一条路。